# Module 4: Hosting a Strands Agent on Amazon Bedrock AgentCore Runtime

## Overview

In this module, we will learn how to **host a Strands agent** on the **Amazon Bedrock AgentCore Runtime**.  
We will use the **Amazon Bedrock AgentCore Python SDK** to deploy a Strands agent that connects to the **MCP Server** created in **Module 3**, enabling natural-language requests to trigger real tool actions.

The **AgentCore SDK** abstracts the underlying agent execution model, allowing you to focus on the agent’s **conversation logic** instead of runtime plumbing.

---

## Module Details

| Information        | Details                                                   |
|-------------------|------------------------------------------------------------|
| **Module type**    | Hosting Agent                                             |
| **Agent type**     | Strands Agent                                             |
| **Components**     | Strands Agent running on AgentCore Runtime                |
| **Vertical**       | Cross-vertical                                            |
| **SDKs Used**      | Amazon BedrockAgentCore Python SDK + Strands SDK          |

---

## Module Architecture

The Strands agent will be deployed to AgentCore Runtime and configured to **call the MCP Server tools** from Module 3 to provide **intelligent e-commerce assistance**.

<div style="text-align:left; margin-top: 10px;">
    <img src="images/agent.png" width="60%"/>
</div>

---

## Key Capabilities Demonstrated

- Creating and configuring **Strands agents**
- Testing agents **locally**
- Deploying agents to **AgentCore Runtime**
- Invoking the deployed agent with **secure authentication**

---

## Select the Correct Kernel

Before running any cells:

1. Click the **Kernel selector** in the top-right corner of the notebook.

<div style="text-align:left; margin-top: 10px;">
    <img src="images/click_kernel.png" width="50%"/>
</div>

2. Select **Python 3 (ipykernel)**.
3. Click **Select** to confirm.

<div style="text-align:left; margin-top: 10px;">
    <img src="images/choose_kernel.png" width="50%"/>
</div>

---

## Next Step

We will begin by:

- Installing packages required to run the Strands agent
- Initializing a **boto3 session** to confirm the execution role

This prepares the environment for deployment to AgentCore Runtime. **The pip install may have pip dependency resolver warnings, that is to be expected and you can continue with the workshop. You can ignore the warnings and continue the workshop.**

---


In [ ]:
!pip install --force-reinstall -U -r requirements_agent.txt --quiet


In [ ]:
import boto3
# This will automatically use ~/.aws/credentials
session = boto3.Session()
region = session.region_name



In [ ]:
!aws sts get-caller-identity

## Run Strands Agent and Invoke MCP Locally

Before deploying and building the AgentCore Runtime we will first test and run a strands agent locally which connects to the MCP server. **Please note you may get a "Session termination failed: 404" error in the middle of the response.** This is happening because when the with block exits, Python calls the context manager's __exit__ method, which attempts to clean up the MCP client session. The 404 error suggests that the session may have already terminated. **You can continue the workshop.**

In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.tools.mcp.mcp_client import MCPClient
import sys
import json


try:
    ssm_client = boto3.client('ssm', region_name=region)
    agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
    agent_arn = agent_arn_response['Parameter']['Value']
    print(f"Retrieved Agent ARN: {agent_arn}")

    secrets_client = boto3.client('secretsmanager', region_name=region)
    response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
    secret_value = response['SecretString']
    parsed_secret = json.loads(secret_value)
    bearer_token = parsed_secret['bearer_token']
    print("✓ Retrieved bearer token from Secrets Manager")
        
except Exception as e:
    print(f"Error retrieving credentials: {e}")
    sys.exit(1)
    
if not agent_arn or not bearer_token:
    print("Error: AGENT_ARN or BEARER_TOKEN not retrieved properly")
    sys.exit(1)

encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
headers = {
    "authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json"
}

print(f"\nConnecting to: {mcp_url}")
print("Headers configured ✓")
streamable_http_mcp_client = MCPClient(lambda: streamablehttp_client(mcp_url, headers=headers))

with streamable_http_mcp_client:

    tools = streamable_http_mcp_client.list_tools_sync()

    agent = Agent(name = "Ecommerce Assistant", model= "global.anthropic.claude-haiku-4-5-20251001-v1:0", tools=tools)

    response = agent("Can you tell me the inventory count for SPRT-001?")
    print(response)

## Host Agent on AgentCore

Now we will host the agent on AgentCore. The following code defines the agent.

In [ ]:
%%writefile agent_agentcore.py
import boto3
import json
import sys
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.tools.mcp.mcp_client import MCPClient
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Dict, Any


app = FastAPI(title="Strands Agent Server", version="1.0.0")
system_prompt = """# E-Commerce Assistant System Prompt

You are an intelligent e-commerce assistant with access to a product catalog and ordering system. Your role is to help customers browse products, check availability, get product information, and place orders.

## Available Tools

You have access to these MCP tools:

1. **search_product_knowledge(query)** - Search product information from knowledge base using natural language queries
2. **list_products()** - Get a complete list of all available products with basic info (ID, name, price, inventory, category)
3. **get_product(product_id)** - Get detailed information about a specific product including description, images, etc.
4. **check_inventory(product_id)** - Check current stock levels and availability for a specific product
5. **place_order(product_id, quantity)** - Place an order for a specific product and quantity

## Tool Usage Strategy

**IMPORTANT**: Always gather necessary information before taking actions. Follow this workflow:

### For Product Inquiries:
- Use `search_product_knowledge()` first for general product questions or when customers describe what they want
- Use `list_products()` to show available options or when customers want to browse
- Use `get_product()` for detailed information about specific products

### For Inventory Questions:
- Use `check_inventory()` to verify stock levels before recommending products
- Always check availability before suggesting quantities

### For Orders:
- **NEVER place orders without first:**
  1. Using `list_products()` or `get_product()` to confirm the exact product_id
  2. Using `check_inventory()` to verify sufficient stock
- Only use `place_order()` after you have the correct product_id and confirmed availability
- If the user says they want to order a specific product and provide the desired quantity, order it - no need to confirm in this case

## Customer Interaction Guidelines

- Be helpful and conversational
- Always verify product details and availability before placing orders
- Provide clear product information including prices and stock levels
- Ask for clarification if customer requests are ambiguous
- Confirm order details before processing
- Handle errors gracefully and suggest alternatives when products are out of stock

## Error Handling

- If a tool returns an error, explain the issue to the customer and suggest alternatives
- If inventory is insufficient, inform the customer of available quantity and ask if they want to order that amount instead
- Always check tool responses for error status before proceeding

Remember: Your goal is to provide excellent customer service while ensuring accurate order processing through proper tool usage sequencing.
"""

def initialize_agent_values():
    """Initialize the agent with MCP tools"""
    boto_session = boto3.Session()
    region = boto_session.region_name
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved MCP Server ARN: {agent_arn}")
        
        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Retrieved bearer token from Secrets Manager")
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    if not agent_arn or not bearer_token:
        print("Error: AGENT_ARN or BEARER_TOKEN not retrieved properly")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to MCP Server: {mcp_url}")
    print("Headers configured ✓")

    return mcp_url, headers


class InvocationRequest(BaseModel):
    input: Dict[str, Any]

class InvocationResponse(BaseModel):
    output: Dict[str, Any]


@app.post("/invocations", response_model=InvocationResponse)
async def invoke_agent(request: InvocationRequest):
    try:
        mcp_url, headers = initialize_agent_values()
        streamable_http_mcp_client = MCPClient(lambda: streamablehttp_client(mcp_url, headers=headers))
    
        with streamable_http_mcp_client:

            tools = streamable_http_mcp_client.list_tools_sync()
            
            agent = Agent(
                name="Ecommerce Assistant", 
                system_prompt=system_prompt,
                model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
                tools=tools
            )
            
            print("✅ Agent initialized successfully")

            user_message = request.input.get("prompt", "")
            if not user_message:
                raise HTTPException(
                    status_code=400,
                    detail="No prompt found in input. Please provide a 'prompt' key in the input."
                )

            result = agent(user_message)
            response = {
                "message": result.message,
                "model": "strands-agent",
            }

            return InvocationResponse(output=response)

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Agent processing failed: {str(e)}")

@app.get("/ping")
async def ping():
    return {"status": "healthy"}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8080)

## Setting up Amazon Cognito for Authentication

AgentCore Runtime requires authentication. We'll use Amazon Cognito to provide JWT tokens for accessing our deployed MCP server.

In [ ]:
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

from utils import setup_cognito_user_pool, create_agentcore_role

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
if cognito_config is not None:
    print("Cognito setup completed ✓")
    print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
    print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

In [ ]:
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            cognito_config['client_id']
        ],
        "discoveryUrl": cognito_config['discovery_url'],
    }
}

## Configure AgentCore Runtime Deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch. Before we do this we will create a basic AgentCore execution role.

During the configure step, your docker file will be generated based on your application code.

<div style="text-align:left">
    <img src="images/configuring.png" width="60%"/>
</div>

In [ ]:
agent_agentcore_role = create_agentcore_role('agent')
agent_role = agent_agentcore_role['Role']['RoleName']

**Note that you will get a yellow warning message from the following cell while configuring the AgentCore Runtime. These can be ignored.** The first message will say you do not have a container engine found. We will not need a container engine locally because when we launch to AgentCore Runtime it will build our container using AWS CodeBuild. You will also see a message saying "Platform mismatch". That is because the jupyter notebook in Amazon SageMaker AI runs on amd64 and AgentCore requires ARM. However, when AWS CodeBuild builds the image it will build it on ARM making it compatible to run on AgentCore.

In [ ]:
# Deploy Agent to AgentCore Runtime
from bedrock_agentcore_starter_toolkit import Runtime

# Create new runtime instance for the agent
agent_agentcore = Runtime()

# Configure - same pattern as MCP server
response = agent_agentcore.configure(
    entrypoint="agent_agentcore.py",
    auto_create_execution_role=False,
    execution_role=agent_role,
    auto_create_ecr=True,
    requirements_file="requirements_agent.txt",
    region=region,
    authorizer_configuration=auth_config,  
    protocol="HTTP",  
    agent_name="strands_ecommerce_agent"
)
print("Agent configuration completed ✓")


## Launching Agent to AgentCore Runtime


Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>


In [ ]:
# Launch agent to AgentCore
print("Launching Strands agent to AgentCore Runtime...")
agent_launch_result = agent_agentcore.launch()
print("Agent launch completed ✓")
print(f"Agent ARN: {agent_launch_result.agent_arn}")


## Checking AgentCore Runtime Status

Now that we've deployed the AgentCore Runtime, let's check for its deployment status and wait for it to be ready:

In [ ]:
# Check status - same as MCP server
print("Checking Agent AgentCore Runtime status...")
status_response = agent_agentcore.status()
status = status_response.endpoint['status']
print(f"Initial status: {status}")

# Wait for READY status
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    print(f"Status: {status} - waiting...")
    time.sleep(10)
    status_response = agent_agentcore.status()
    status = status_response.endpoint['status']

if status == 'READY':
    print("✅ Agent AgentCore Runtime is READY!")
else:
    print(f"⚠ Agent AgentCore Runtime status: {status}")


## Permissions for the Execution Role

The agent will need to be able to authenticate its requests to the MCP Server running on AgentCore Runtime so it can invoke the MCP tools. In order to do this we need to add permissions to the agent's execution role. It will need to access the SSM parameter and the secret we saved in the previous module for the MCP Server. In the next few cells we are going to define an additional IAM policy and add it to the execution role of the agent.

In [ ]:
sts = session.client('sts')
account_id = sts.get_caller_identity()['Account']

In [ ]:
import json

iam_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "secretsmanager:GetSecretValue",
                "secretsmanager:DescribeSecret"
            ],
            "Resource": f"arn:aws:secretsmanager:{region}:{account_id}:secret:mcp_server/cognito/credentials-*" #ADD THE SECRET FOR THE MCP SERVER
        },
        {
            "Effect": "Allow",
            "Action": [
                "ssm:GetParameter",
                "ssm:GetParameters"
            ],
            "Resource": f"arn:aws:ssm:{region}:{account_id}:parameter/mcp_server/runtime/agent_arn" #ADD THE SSM PARAMETER FOR THE MCP SERVER
        }
    ]
}

iam_agent_policy = json.dumps(iam_policy, indent=2)

Now that we have the proper IAM policy to allow access to the AWS resources we will update the execution role with the policy in the following code block. 

In [ ]:
import boto3

iam = boto3.client('iam')

iam.put_role_policy(
    RoleName = agent_role,
    PolicyName = 'agent_resource_policy',
    PolicyDocument = iam_agent_policy
)

## Storing Configuration for Remote Access

Before we can invoke our deployed MCP server, let's store the Agent ARN and Cognito configuration in AWS Systems Manager Parameter Store and AWS Secrets Manager for easy retrieval later when we want to invoke the agent from our application:

In [ ]:
import boto3
import json

ssm_client = boto3.client('ssm', region_name=region)
secrets_client = boto3.client('secretsmanager', region_name=region)

try:
    cognito_credentials_response = secrets_client.create_secret(
        Name='agent/cognito/credentials',
        Description='Cognito credentials for agent',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials stored in Secrets Manager")
except secrets_client.exceptions.ResourceExistsException:
    secrets_client.update_secret(
        SecretId='agent/cognito/credentials',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials updated in Secrets Manager")

agent_arn_response = ssm_client.put_parameter(
    Name='/agent/runtime/agent_arn',
    Value=agent_launch_result.agent_arn,
    Type='String',
    Description='Agent ARN ',
    Overwrite=True
)
print("✓ Agent ARN stored in Parameter Store")

print("\nConfiguration stored successfully!")
print(f"Agent ARN: {agent_launch_result.agent_arn}")

## Creating Remote Testing Client

Now let's create a client to test our deployed MCP server. This client will retrieve the necessary credentials from AWS and connect to the deployed server. The client will be able to invoke the agent with natural language and autonomously call the necessary MCP Server tools.

<div style="text-align:left">
    <img src="images/invoke.png" width="50%"/>
</div>

In [ ]:
%%writefile invoke_full_agent.py
import boto3
import json
import sys
import requests
import urllib.parse
import uuid
from boto3.session import Session
import time

def main():
    boto_session = Session()
    region = boto_session.region_name

    print(f"Using AWS region: {region}")

    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/agent/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='agent/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Retrieved bearer token from Secrets Manager")

    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)

    # URL encode the agent ARN
    escaped_agent_arn = urllib.parse.quote(agent_arn, safe='')
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"
    
    print(f"AgentCore URL: {url}")

    print(f"\n🤖 Testing AgentCore Runtime Agent with OAuth:")
    print("=" * 50)

    # Test queries (natural language)
    test_queries = [
        "tell me about the running shoes",
        "check inventory for product SPRT-001", 
        "list all available products",
        "place an order for 1 unit of product SPRT-001",
        "check inventory for SPRT-001 after the order",
        "get detailed information about product SPRT-001"
    ]

    for i, query in enumerate(test_queries, 1):
        try:
            print(f"\n➕ Test {i}: {query}")
            
            # Generate session ID (must be 33+ characters)
            session_id = f"test-session-{uuid.uuid4()}"
            
            # Set up OAuth headers
            headers = {
                "Authorization": f"Bearer {bearer_token}",
                "Content-Type": "application/json",
            }       

            invoke_response = requests.post(
                url,
                headers=headers,
                data=json.dumps({"input": {"prompt":query}}),
                timeout=120
            )

            # Handle response
            print(f"Status Code: {invoke_response.status_code}")
            time.sleep(5)
            
            if invoke_response.status_code == 200:
                try:
                    response_data = invoke_response.json()
                    print("✅ Response JSON:")
                    print(json.dumps(response_data, indent=2))
                except json.JSONDecodeError:
                    print("✅ Response Text:")
                    print(invoke_response.text)
                    
            elif invoke_response.status_code >= 400:
                print(f"❌ Error Response ({invoke_response.status_code}):")
                try:
                    error_data = invoke_response.json()
                    print(json.dumps(error_data, indent=2))
                except json.JSONDecodeError:
                    print(invoke_response.text)
            else:
                print(f"⚠️ Unexpected status code: {invoke_response.status_code}")
                print("Response text:")
                print(invoke_response.text[:500])

        except Exception as e:
            print(f"   Error: {e}")

        print("-" * 30)

    print("\n✅ Agent testing completed!")

if __name__ == "__main__":
    main()

## Testing Your Deployed Agent

Now we can invoke the agent. Reminder, if you did not add the IAM permissions to the execution role for the agent this step will have an error. Also, one of the tools we will test here will actually order a product for us. So if you go back to the web UI after running the following cell, you will see the order in the "Order History" tab.

In [ ]:
import time 
time.sleep(10) #adding in a short sleep period to ensure IAM roles have propogated 

print("Testing agent invocation...")
print("=" * 50)
!python invoke_full_agent.py

## Lambda Invocation

We now have a strands agent running on AgentCore Runtime that can invoke a MCP server which is also running on AgentCore Runtime. However, our end goal is to invoke this from the front end of our web application. To do that we will use two lambda functions. One lambda function that is triggered from the API Gateway, invokes the processor lambda, and then returns. Then the processor lambda will run and invoke the agent and respond to the API Gateway.

Both of these lambda functions and API Gateway were provisioned in the original cloud formation template that deployed when the workshop account was created. The lambda functions were deployed with simple template code that needs to be replaced with the proper functionality. In the following cell blocks we will redeploy each lambda function. We will also need to add permissions to the Processor lambda execution role to read from SSM and Secrets Manager so it can get the proper credentials to invoke the agent. 

<div style="text-align:left">
    <img src="images/lambda.png" width="50%"/>
</div>

The following are the two lambda function names that we need to update the code for. 

In [ ]:
proccessor_name = "secure-ecommerce-stack-processer-lambda"
websocket_name = "secure-ecommerce-stack-mcp-chatbot"

We will use this function to update the lambda file code.

In [ ]:
import boto3
import zipfile
import os
from io import BytesIO

def update_lambda_from_file(function_name, python_file_path, region):
    """
    Update an existing Lambda function with code from a Python file.
    This preserves all existing Lambda configuration (API Gateway, environment variables, etc.)
    and only updates the function code.
    """
    lambda_client = boto3.client('lambda', region_name=region)
    
    try:
        # Read the Python file
        with open(python_file_path, 'r') as file:
            python_code = file.read()
        # Create a ZIP file in memory with the Python code
        zip_buffer = BytesIO()
        
        with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zip_file:
            # Add the Python file to the ZIP
            # Use 'lambda_function.py' as the filename in the ZIP (standard Lambda entry point)
            zip_file.writestr('lambda_function.py', python_code)
        zip_buffer.seek(0)
        
        # Update the Lambda function code
        response = lambda_client.update_function_code(
            FunctionName=function_name,
            ZipFile=zip_buffer.read()
        )
        
        print(f"✅ Successfully updated Lambda function: {function_name}")
        print(f"📦 Version: {response['Version']}")
        print(f"🕒 Last Modified: {response['LastModified']}")
        print(f"⚡ Runtime: {response['Runtime']}")
        print(f"🎯 Handler: {response['Handler']}")
        
        return response
        
    except FileNotFoundError:
        print(f"❌ Error: File '{python_file_path}' not found")
        return None
    except lambda_client.exceptions.ResourceNotFoundException:
        print(f"❌ Error: Lambda function '{function_name}' not found")
        return None
    except Exception as e:
        print(f"❌ Error updating Lambda function: {str(e)}")
        return None


## Lambda for Web Socket

Now we will write the websocket_lambda.py file. 

In [ ]:
%%writefile websocket_lambda.py
import json
import boto3
import os

def lambda_handler(event, context):
    print(f"Event: {json.dumps(event)}")
    
    # Get WebSocket API management client
    api_gateway_management = boto3.client(
        'apigatewaymanagementapi',
        endpoint_url=os.environ['WEBSOCKET_API_ENDPOINT']
    )
    
    try:
        route_key = event.get('requestContext', {}).get('routeKey')
        connection_id = event.get('requestContext', {}).get('connectionId')
        
        if route_key == '$connect':
            print(f"Client connected: {connection_id}")
            return {'statusCode': 200}
        
        elif route_key == '$disconnect':
            print(f"Client disconnected: {connection_id}")
            return {'statusCode': 200}
        
        elif route_key == 'chat':
            # Parse the message
            body = json.loads(event.get('body', '{}'))
            user_message = body.get('message', 'No message provided')
            print(f"Processing message from {connection_id}: {user_message}")
            
            # Invoke the background processing Lambda asynchronously
            lambda_client = boto3.client('lambda')
            
            payload = {
                'connection_id': connection_id,
                'user_message': user_message,
                'websocket_endpoint': os.environ['WEBSOCKET_API_ENDPOINT']
            }
            
            try:
                lambda_client.invoke(
                    FunctionName=os.environ['BACKGROUND_PROCESSOR_FUNCTION_NAME'],  
                    InvocationType='Event',  
                    Payload=json.dumps(payload)
                )
                
                print(f"Successfully triggered background processing for connection {connection_id}")
                
            except Exception as invoke_error:
                print(f"Error invoking background processor: {str(invoke_error)}")
                api_gateway_management.post_to_connection(
                    ConnectionId=connection_id,
                    Data=json.dumps({
                        'type': 'error',
                        'message': f'Failed to start background processing: {str(invoke_error)}'
                    })
                )
            
            return {'statusCode': 200}
        
        else:
            return {'statusCode': 400, 'body': 'Unknown route'}
            
    except Exception as e:
        print(f"Error: {str(e)}")
        try:
            if connection_id:
                api_gateway_management.post_to_connection(
                    ConnectionId=connection_id,
                    Data=json.dumps({
                        'type': 'error',
                        'message': f'Error processing request: {str(e)}'
                    })
                )
        except:
            pass
        return {'statusCode': 500}


Next, we will push the code to the lambda function. We will also then make sure the configuration is set for the proper lambda handler. Before doing this we will wait 5 seconds to let the lambda function deploy before trying to update the configuration. 

In [ ]:
update_lambda_from_file(
    function_name=websocket_name,
    python_file_path="websocket_lambda.py",
    region = region
)

In [ ]:
import time
time.sleep(5)

In [ ]:
lambda_client = boto3.client('lambda', region_name=region)
lambda_client.update_function_configuration(
            FunctionName=websocket_name,
            Handler='lambda_function.lambda_handler')

## Lambda Processor

Now we will go through the same process for the lambda processor

In [ ]:
%%writefile processor_lambda.py
import json
import boto3
import time
import os
import sys
import urllib.request
import urllib.parse
import urllib.error
import uuid
from boto3.session import Session

def invoke_agent(query, region=None):
    """Invoke a single agent query and return the response"""
    if not region:
        boto_session = Session()
        region = boto_session.region_name
    
    try:
        # Get agent ARN from SSM
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/agent/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        
        # Get bearer token from Secrets Manager
        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='agent/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        
    except Exception as e:
        return {
            'success': False,
            'error': f'Error retrieving credentials: {str(e)}',
            'status_code': 500
        }
    
    # Build URL
    escaped_agent_arn = urllib.parse.quote(agent_arn, safe='')
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"
    
    try:
        # Prepare request data
        data = json.dumps({"input": {"prompt": query}}).encode('utf-8')
        
        # Create request
        req = urllib.request.Request(
            url,
            data=data,
            headers={
                "Authorization": f"Bearer {bearer_token}",
                "Content-Type": "application/json",
            },
        )
        
        # Make request
        with urllib.request.urlopen(req, timeout=120) as response:
            status_code = response.getcode()
            response_data = response.read().decode('utf-8')
            
            if status_code == 200:
                try:
                    parsed_response = json.loads(response_data)
                    return {
                        'success': True,
                        'data': parsed_response,
                        'status_code': status_code,
                        'query': query
                    }
                except json.JSONDecodeError:
                    return {
                        'success': True,
                        'data': response_data,
                        'status_code': status_code,
                        'query': query
                    }
            else:
                return {
                    'success': False,
                    'error': f'Unexpected status code: {status_code}',
                    'data': response_data[:500],
                    'status_code': status_code,
                    'query': query
                }
                
    except urllib.error.HTTPError as e:
        try:
            error_data = e.read().decode('utf-8')
            try:
                parsed_error = json.loads(error_data)
                error_message = parsed_error
            except json.JSONDecodeError:
                error_message = error_data
        except:
            error_message = str(e)
            
        return {
            'success': False,
            'error': f'HTTP Error: {e.code}',
            'data': error_message,
            'status_code': e.code,
            'query': query
        }
        
    except urllib.error.URLError as e:
        return {
            'success': False,
            'error': f'URL Error: {str(e)}',
            'status_code': 500,
            'query': query
        }
        
    except Exception as e:
        return {
            'success': False,
            'error': f'Unexpected error: {str(e)}',
            'status_code': 500,
            'query': query
        }

def extract_query_and_response(json_body):
    """Extract the original query and the assistant's response from Lambda result"""
    try:
        # Parse JSON if it's a string
        if isinstance(json_body, str):
            data = json.loads(json_body)
        else:
            data = json_body
        
        # Extract the query
        query = data.get('query', 'No query found')
        
        # Extract the response content
        response_text = None
        if data.get('success') and 'data' in data:
            output = data['data'].get('output', {})
            message = output.get('message', {})
            content = message.get('content', [])
            
            # Get the text from the first content item
            if content and len(content) > 0:
                response_text = content[0].get('text', 'No response text found')
        
        return {
            'query': query,
            'response': response_text,
            'success': data.get('success', False),
            'status_code': data.get('status_code'),
            'model': data.get('data', {}).get('output', {}).get('model'),
            'raw_data': data
        }
        
    except json.JSONDecodeError as e:
        return {
            'query': None,
            'response': None,
            'success': False,
            'error': f'JSON decode error: {str(e)}',
            'raw_data': json_body
        }
        
    except Exception as e:
        return {
            'query': None,
            'response': None,
            'success': False,
            'error': f'Extraction error: {str(e)}',
            'raw_data': json_body
        }

def send_websocket_message(api_gateway_management, connection_id, message_data):
    """Helper function to send WebSocket message with error handling"""
    try:
        api_gateway_management.post_to_connection(
            ConnectionId=connection_id,
            Data=json.dumps(message_data)
        )
        return True
    except api_gateway_management.exceptions.GoneException:
        print(f"Connection {connection_id} is gone")
        return False
    except Exception as e:
        print(f"Error sending WebSocket message: {str(e)}")
        return False

def lambda_handler(event, context):
    print(f"Background processor event: {json.dumps(event)}")
    
    # Extract data from the event
    connection_id = event.get('connection_id')
    user_message = event.get('user_message')
    websocket_endpoint = event.get('websocket_endpoint')
    
    if not all([connection_id, user_message, websocket_endpoint]):
        print("Missing required parameters")
        return {'statusCode': 400, 'body': 'Missing required parameters'}
    
    # Create WebSocket API management client
    api_gateway_management = boto3.client(
        'apigatewaymanagementapi',
        endpoint_url=websocket_endpoint
    )
    
    try:
        
        # Invoke the agent 
        print(f"Starting agent invocation for: {user_message}")
        result = invoke_agent(user_message)
        print(f'Agent result: {result}')
        
        # Extract the response
        extracted_result = extract_query_and_response(result)
        
        # Send final response
        if extracted_result['success'] and extracted_result['response']:
            final_response = extracted_result['response']
            response_type = 'response'
        else:
            final_response = f"Agent error: {extracted_result.get('error', 'Unknown error occurred')}"
            response_type = 'error'
        
        send_websocket_message(api_gateway_management, connection_id, {
            'type': response_type,
            'message': final_response,
            'timestamp': time.time(),
            'query': user_message
        })
        
        print(f"Successfully processed request for connection {connection_id}")
        return {'statusCode': 200, 'body': 'Processing completed'}
        
    except Exception as e:
        print(f"Error in background processing: {str(e)}")
        
        # Send error message to client
        send_websocket_message(api_gateway_management, connection_id, {
            'type': 'error',
            'message': f'Background processing failed: {str(e)}'
        })
        
        return {'statusCode': 500, 'body': f'Error: {str(e)}'}


Next, we will push the code to the lambda function. We will also then make sure the configuration is set for the proper lambda handler. Before doing this we will wait 5 seconds to let the lambda function deploy before trying to update the configuration. 

In [ ]:
update_lambda_from_file(
    function_name=proccessor_name,  
    python_file_path="processor_lambda.py",
    region = region  
)

In [ ]:
import time
time.sleep(5)

In [ ]:
lambda_client = boto3.client('lambda', region_name=region)
lambda_client.update_function_configuration(
            FunctionName=proccessor_name,
            Handler='lambda_function.lambda_handler')

## Lambda Permissions

In order for the Lambda to invoke the agent (similar to the agent invoking the MCP Server) we need proper permissions. This time the Lambda needs the secret and SSM Parameter for the agent to send the information in a header during the request to the agent. We will now add the those permissions.

The following is the exeuction role of for the processor lambda.

In [ ]:
lambda_processor_execution_role_arn = "secure-ecommerce-stack-processer-lambda-role"

Now we will create the IAM policy.

In [ ]:
sts = session.client('sts')
account_id = sts.get_caller_identity()['Account']

In [ ]:
import json

iam_lambda_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "secretsmanager:GetSecretValue",
                "secretsmanager:DescribeSecret"
            ],
            "Resource": f"arn:aws:secretsmanager:{region}:{account_id}:secret:agent/cognito/credentials-*" 
        },
        {
            "Effect": "Allow",
            "Action": [
                "ssm:GetParameter",
                "ssm:GetParameters"
            ],
            "Resource": f"arn:aws:ssm:{region}:{account_id}:parameter/agent/runtime/agent_arn" 
        }
    ]
}

iam_lambda_policy = json.dumps(iam_lambda_policy, indent=2)

Now that we have the proper IAM policy to allow access to the AWS resources we will update the execution role with the policy in the following code block. 

In [ ]:
import boto3

iam = boto3.client('iam')

iam.put_role_policy(
    RoleName = lambda_processor_execution_role_arn,
    PolicyName = 'lambda_access',
    PolicyDocument = iam_lambda_policy
)

# 🎉 Congratulations!

You have successfully:

✅ **Created an MCP server** with custom tools  
✅ **Created an Agent** that connects to the MCP server  

**You are now ready to move on to the next module!** Please return to the **workshop instructions**!

---

---

---

### Reauthenticate User (Don't Recreate the User Pool)

**Only use the following code if your tokens have expired.** The bearer token used to authenticate the request to AgentCore Runtime was set up to be valid for six hours, so if this time limit expires you will need to refresh the tokens to invoke the agent. This code only refreshes the strands agent token, the code to reauthenticate the MCP server token is in module 3. 

### ⚠️ IMPORTANT – Manual Step Required

Copy over the client_id and region from the output earlier in the notebook when cognito was set up.

In [ ]:
from utils import reauthenticate_user
import boto3
import json

client_id = '' #paste in from the output above 
region = ''
new_bearer_token = reauthenticate_user(client_id)

secrets_client = boto3.client('secretsmanager', region_name=region)

try:
    secret_name='agent/cognito/credentials'
    current_secret_response = secrets_client.get_secret_value(SecretId=secret_name)
    current_secret = json.loads(current_secret_response['SecretString'])
    
    # Update only the bearer_token field
    current_secret['bearer_token'] = new_bearer_token
    
    # Update the secret
    secrets_client.update_secret(
        SecretId=secret_name,
        SecretString=json.dumps(current_secret)
    )
    
    print(f"✓ Token refreshed and updated in Secrets Manager: {secret_name}")
    
except Exception as e:
    print(f"Error updating token in Secrets Manager: {e}")
    raise